In [ ]:
!pip install duckdb pandas matplotlib seaborn langgraph langchain-openai langchain-core openai python-dotenv graphviz

In [ ]:
# Importações necessárias
import os
import duckdb  # Banco de dados embutido
import pandas as pd  # Manipulação de dados
import numpy as np  # Operações numéricas
import matplotlib.pyplot as plt  # Visualizações
import seaborn as sns  # Visualizações aprimoradas
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END  # Para criar o fluxo de trabalho
from langchain_openai.chat_models import ChatOpenAI  # Modelo de linguagem
from langchain_core.prompts import ChatPromptTemplate  # Para criar prompts personalizados
from dotenv import load_dotenv  # Para carregar variáveis de ambiente

# Carregar variáveis de ambiente
OPENAI_API_KEY = "sk-KEY"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)



In [7]:
# Conectar ao banco de dados em memória
conn = duckdb.connect(database=':memory:')

# Criar dados fictícios de inventário
inventory_data = pd.DataFrame({
    'product_id': [f'P{str(i).zfill(3)}' for i in range(1, 21)],
    'product_name': [f'Produto {i}' for i in range(1, 21)],
    'current_stock': np.random.randint(0, 100, size=20),
    'reorder_level': np.random.randint(10, 50, size=20),
    'lead_time_days': np.random.randint(1, 15, size=20)
})

# Criar dados fictícios de vendas históricas
date_range = pd.date_range(end=pd.Timestamp.today(), periods=180)
sales_history = pd.DataFrame({
    'date': np.random.choice(date_range, size=500),
    'product_id': np.random.choice(inventory_data['product_id'], size=500),
    'quantity_sold': np.random.randint(1, 10, size=500)
})

# Inserir os dados no DuckDB
conn.execute("CREATE TABLE inventory AS SELECT * FROM inventory_data")
conn.execute("CREATE TABLE sales_history AS SELECT * FROM sales_history")

In [8]:
#testando o banco de dados
print(conn.execute("SELECT * FROM inventory").df())
print(conn.execute("SELECT * FROM sales_history").df())

   product_id product_name  current_stock  reorder_level  lead_time_days
0        P001    Produto 1             43             17               7
1        P002    Produto 2             59             25              10
2        P003    Produto 3             62             33              10
3        P004    Produto 4             36             18              10
4        P005    Produto 5             64             28               8
5        P006    Produto 6             73             39               4
6        P007    Produto 7             22             32               6
7        P008    Produto 8             26             14               4
8        P009    Produto 9             19             20               7
9        P010   Produto 10             44             19               1
10       P011   Produto 11             19             17               7
11       P012   Produto 12             88             45               8
12       P013   Produto 13             24          

In [9]:
# Estrutura do estado
class State(TypedDict):
    low_stock_products: pd.DataFrame
    demand_forecast: pd.DataFrame
    replenishment_orders: pd.DataFrame
    alerts: List[str]
    errors: List[str]

In [10]:
def check_inventory_levels(state: State) -> State:
    """
    Verifica o nível atual de estoque e identifica produtos abaixo do nível de reabastecimento.
    """
    try:
        # Consultar o banco de dados para obter o inventário atual
        inventory = conn.execute("SELECT * FROM inventory").df()
        
        # Identificar produtos com estoque abaixo do nível de reabastecimento
        low_stock = inventory[inventory['current_stock'] <= inventory['reorder_level']]
        
        # Atualizar o estado
        state['low_stock_products'] = low_stock
    except Exception as e:
        state['errors'].append(f"Erro ao verificar os níveis de inventário: {e}")
    return state

In [11]:
def forecast_demand(state: State) -> State:
    """
    Prevê a demanda futura com base no histórico de vendas.
    """
    try:
        # Obter produtos com baixo estoque
        low_stock = state.get('low_stock_products')
        
        if low_stock is None or low_stock.empty:
            state['errors'].append("Nenhum produto com baixo estoque para prever demanda.")
            return state
        
        # Obter histórico de vendas
        sales = conn.execute("SELECT * FROM sales_history").df()
        
        # Combinar dados
        data = sales.merge(low_stock[['product_id']], on='product_id')
        
        # Prever demanda (neste exemplo simples, usaremos a média diária)
        forecast = data.groupby('product_id').agg({
            'quantity_sold': lambda x: x.sum() / 180  # Média diária nos últimos 180 dias
        }).reset_index()
        forecast.rename(columns={'quantity_sold': 'daily_average_sold'}, inplace=True)
        
        # Atualizar o estado
        state['demand_forecast'] = forecast
    except Exception as e:
        state['errors'].append(f"Erro ao prever a demanda: {e}")
    return state

In [12]:
def generate_replenishment_orders(state: State) -> State:
    """
    Gera ordens de reabastecimento com base na previsão de demanda e tempo de entrega.
    """
    try:
        low_stock = state.get('low_stock_products')
        forecast = state.get('demand_forecast')
        
        if low_stock is None or low_stock.empty or forecast is None or forecast.empty:
            state['errors'].append("Dados insuficientes para gerar ordens de reabastecimento.")
            return state
        
        # Combinar dados
        orders = low_stock.merge(forecast, on='product_id')
        
        # Calcular a quantidade necessária
        orders['replenishment_quantity'] = (
            orders['daily_average_sold'] * orders['lead_time_days']
        ).astype(int) - orders['current_stock']
        
        # Filtrar pedidos com quantidade positiva
        orders = orders[orders['replenishment_quantity'] > 0]
        
        # Atualizar o estado
        state['replenishment_orders'] = orders[['product_id', 'product_name', 'replenishment_quantity']]
    except Exception as e:
        state['errors'].append(f"Erro ao gerar ordens de reabastecimento: {e}")
    return state

In [13]:
def send_alerts_or_place_orders(state: State) -> State:
    """
    Envia alertas ou faz pedidos de reabastecimento.
    """
    try:
        orders = state.get('replenishment_orders')
        
        if orders is None or orders.empty:
            state['alerts'].append("Nenhum produto necessita de reabastecimento imediato.")
            return state
        
        # Exemplo: Enviar um alerta para cada produto
        for _, row in orders.iterrows():
            alert = f"Produto {row['product_name']} (ID: {row['product_id']}) precisa ser reabastecido. Quantidade sugerida: {row['replenishment_quantity']} unidades."
            state['alerts'].append(alert)
            # Aqui, você poderia integrar com um sistema de pedidos automático
    except Exception as e:
        state['errors'].append(f"Erro ao enviar alertas ou fazer pedidos: {e}")
    return state

In [14]:
# Inicializar o grafo de estado
workflow = StateGraph(State)

# Adicionar nós
workflow.add_node("check_inventory_levels", check_inventory_levels)
workflow.add_node("forecast_demand", forecast_demand)
workflow.add_node("generate_replenishment_orders", generate_replenishment_orders)
workflow.add_node("send_alerts_or_place_orders", send_alerts_or_place_orders)

# Definir as arestas
workflow.add_edge(START, "check_inventory_levels")
workflow.add_edge("check_inventory_levels", "forecast_demand")
workflow.add_edge("forecast_demand", "generate_replenishment_orders")
workflow.add_edge("generate_replenishment_orders", "send_alerts_or_place_orders")
workflow.add_edge("send_alerts_or_place_orders", END)

# Definir o ponto de entrada
workflow.set_entry_point("check_inventory_levels")

# Compilar o grafo
app = workflow.compile()

In [15]:
print(dir(workflow))

['__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_add_schema', '_all_edges', '_node_defaults', 'add_conditional_edges', 'add_edge', 'add_node', 'add_sequence', 'branches', 'channels', 'compile', 'compiled', 'context_schema', 'edges', 'input_schema', 'managed', 'nodes', 'output_schema', 'schemas', 'set_conditional_entry_point', 'set_entry_point', 'set_finish_point', 'set_node_defaults', 'state_schema', 'validate', 'waiting_edges']


In [16]:
def manage_inventory():
    """
    Executa o fluxo de gerenciamento de inventário.
    
    Returns:
        State: Estado final após o processamento.
    """
    initial_state = State(
        low_stock_products=None,
        demand_forecast=None,
        replenishment_orders=None,
        alerts=[],
        errors=[]
    )
    final_state = app.invoke(initial_state)
    return final_state

In [17]:
# Executar o gerenciamento de inventário
final_state = manage_inventory()

C:\Users\eugen\AppData\Local\Temp\ipykernel_87748\185573427.py:23: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  forecast.rename(columns={'quantity_sold': 'daily_average_sold'}, inplace=True)


In [18]:
# Verificar se houve erros
if final_state['errors']:
    print("Erros encontrados:")
    for error in final_state['errors']:
        print(f"- {error}")
else:
    # Exibir alertas
    if final_state['alerts']:
        print("Alertas:")
        for alert in final_state['alerts']:
            print(f"- {alert}")
    else:
        print("Nenhum alerta a ser exibido.")

Alertas:
- Nenhum produto necessita de reabastecimento imediato.
